# PetFinder · v3 · Regresión más umbrales

**Qué agrega sobre v2.** El target tiene orden (0 es más rápido que 4) y la métrica castiga la distancia al cuadrado. Un clasificador trata las cinco clases como etiquetas sin relación. Acá:

1. En `correr_cv` el modelo pasa a ser `LGBMRegressor`: mismo LightGBM, objetivo de error cuadrático. Devuelve un número continuo por mascota. **El loop no cambia.**
2. Ese número se **corta** en cinco clases con cuatro umbrales elegidos por **cuantiles**, para que la distribución de clases predicha copie la del train.

**Pregunta que responde.** ¿Cuánto vale respetar el orden del target? En esta competencia solía ser el salto más grande por línea de código.

**Por qué no sirven los umbrales 0,5 / 1,5 / 2,5 / 3,5.** El regresor comprime hacia el medio: casi nunca predice menos de 0,5 ni más de 3,5, y las clases 0 y 4 quedan vacías aunque la 4 sea el 28% de los datos. Se compara abajo.

**Para investigar.** Hay otras formas de elegir los umbrales: optimizar kappa directamente sobre las predicciones out-of-fold (`OptimizedRounder` con Nelder-Mead) o buscarlos uno por uno. Hay una celda opcional al final.

**Cómo correrla en Kaggle.** Copy & Edit, confirmar que en *Input* está la competencia *PetFinder.my Adoption Prediction*, *Internet off* (es obligatorio para poder enviar), *Save Version → Save & Run All*. Cuando termina, pestaña *Output → Submit to Competition*. Anotar el kappa out-of-fold que imprime y el score del leaderboard en la tabla del final.

In [1]:
import os
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.model_selection import KFold, StratifiedGroupKFold
from sklearn.metrics import cohen_kappa_score

from tqdm.notebook import tqdm

INPUT = os.environ.get("PETFINDER_INPUT", "/kaggle/input/petfinder-adoption-prediction")
if not os.path.exists(INPUT):   # las notebooks creadas por la API montan la competencia bajo /kaggle/input/competitions/
    INPUT = "/kaggle/input/competitions/petfinder-adoption-prediction"
SEED = 42

In [13]:
train = pd.read_csv(f"{INPUT}/train/train.csv", index_col="PetID")
test = pd.read_csv(f"{INPUT}/test/test.csv", index_col="PetID")
ss = pd.read_csv(f"{INPUT}/test/sample_submission.csv", index_col="PetID")

label = "AdoptionSpeed"

In [14]:
def agregar_features(data):
    """Las tres features de la consigna. Se aplica igual a train y a test."""
    data = data.copy()
    # 1) cuántas publicaciones tiene el rescatista (calculado dentro de cada dataset)
    data["rescuer_count"] = data["RescuerID"].map(data["RescuerID"].value_counts())
    # 2) la mascota no tiene nombre (vacío, "No Name", "No Name Yet"...)
    nombre = data["Name"].fillna("").str.lower().str.replace(" ", "", regex=False)
    data["sin_nombre"] = ((nombre == "") | nombre.str.startswith("noname")).astype(int)
    # 3) largo de la descripción
    desc = data["Description"].fillna("").astype(str)
    data["desc_len"] = desc.str.len()
    data["desc_words"] = desc.str.split().str.len()
        # 4) vacunado y desparasitado (1 = Yes en ambas columnas; 2 = No y 3 = Not Sure cuentan como no)
    data["vac_y_desp"] = ((data["Vaccinated"] == 1) & (data["Dewormed"] == 1)).astype(int)
    return data


train = agregar_features(train)
test = agregar_features(test)

X_train = train.drop(label, axis=1).select_dtypes(exclude="O")
y_train = train[label]
X_test = test[X_train.columns]
groups = train["RescuerID"]   # para que cada rescatista caiga en un solo fold

print(X_train.shape, X_test.shape)

(14993, 24) (3972, 24)


In [17]:
def correr_cv(crear_modelo, X_train, y_train, X_test, groups):
    """El mismo loop del baseline, metido en una función para poder llamarlo varias veces.

    Devuelve valid_preds (una predicción por mascota de train, hecha por el modelo que NO la vio)
    y test_preds (un DataFrame con las 5 predicciones de test, una por fold).
    """
    valid_preds = []
    test_preds = []
    folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    for train_index, valid_index in tqdm(folds.split(X_train, y_train, groups), total=5):
        Xt = X_train.iloc[train_index]
        yt = y_train.iloc[train_index]
        Xv = X_train.iloc[valid_index]

        model = crear_modelo()
        model.fit(Xt, yt)

        valid_preds.append(pd.Series(model.predict(Xv), index=Xv.index, name=label))
        test_preds.append(pd.Series(model.predict(X_test), index=X_test.index, name=label))

    valid_preds = pd.concat(valid_preds).loc[X_train.index]
    test_preds = pd.concat(test_preds, axis=1)
    return valid_preds, test_preds


def kappa(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")

In [19]:
def umbrales_por_cuantiles(pred_cont, y):
    """Cuatro cortes para que la distribución de clases predicha copie la del train."""
    acumulado = np.cumsum(np.bincount(y, minlength=5) / len(y))[:-1]   # 2,7% · 23,3% · 50,2% · 72,0%
    return np.quantile(pred_cont, acumulado)


def cortar(pred_cont, umbrales):
    """Convierte la predicción continua en clase 0..4 con los umbrales."""
    return np.digitize(pred_cont, np.sort(umbrales))

In [20]:
# Parámetros ganadores de v2 (kappa OOF 0,3426 sobre 12 combinaciones). Si cambiás la grilla, pegá acá los nuevos.
BEST_PARAMS = dict(n_estimators=300, learning_rate=0.03, num_leaves=15, colsample_bytree=0.6,
                   random_state=SEED, verbose=-1)


## Referencia: el clasificador de v2 con estos mismos parámetros

In [21]:
valid_clf, test_clf = correr_cv(lambda: LGBMClassifier(**BEST_PARAMS), X_train, y_train, X_test, groups)
kappa_clf = kappa(y_train, valid_clf)
print(f"kappa out-of-fold clasificador: {kappa_clf:.4f}")

  0%|          | 0/5 [00:00<?, ?it/s]

kappa out-of-fold clasificador: 0.3406


## Regresor: la misma función, otro modelo. La predicción ahora es un número continuo

In [22]:
valid_preds, test_preds = correr_cv(lambda: LGBMRegressor(**BEST_PARAMS), X_train, y_train, X_test, groups)
print("predicción continua out-of-fold, resumen:")
print(valid_preds.describe().round(3))

  0%|          | 0/5 [00:00<?, ?it/s]

predicción continua out-of-fold, resumen:
count    14993.000
mean         2.513
std          0.468
min          1.285
25%          2.155
50%          2.466
75%          2.854
max          4.110
Name: AdoptionSpeed, dtype: float64


## Umbrales ingenuos vs. umbrales por cuantiles

In [23]:
ingenuos = np.array([0.5, 1.5, 2.5, 3.5])
umbrales = umbrales_por_cuantiles(valid_preds, y_train)

kappa_ingenuo = kappa(y_train, cortar(valid_preds, ingenuos))
kappa_cuantiles = kappa(y_train, cortar(valid_preds, umbrales))

print("umbrales ingenuos  :", ingenuos.round(3), f"-> kappa {kappa_ingenuo:.4f}")
print("umbrales cuantiles :", umbrales.round(3), f"-> kappa {kappa_cuantiles:.4f}")
print()
print("clases predichas con umbrales ingenuos :", np.bincount(cortar(valid_preds, ingenuos), minlength=5))
print("clases predichas con umbrales cuantiles:", np.bincount(cortar(valid_preds, umbrales), minlength=5))
print("clases reales                          :", np.bincount(y_train, minlength=5))

umbrales ingenuos  : [0.5 1.5 2.5 3.5] -> kappa 0.2700
umbrales cuantiles : [1.745 2.134 2.47  2.799] -> kappa 0.3918

clases predichas con umbrales ingenuos : [   0   43 7820 6856  274]
clases predichas con umbrales cuantiles: [ 410 3090 4037 3259 4197]
clases reales                          : [ 410 3090 4037 3259 4197]


## Resumen de la versión

In [24]:
pd.DataFrame({
    "variante": ["clasificador (v2)", "regresor + umbrales ingenuos", "regresor + umbrales por cuantiles"],
    "kappa_oof": [kappa_clf, kappa_ingenuo, kappa_cuantiles],
}).round(4)

,variante,kappa_oof
0,clasificador (v2),0.3406
1,regresor + umbrales ingenuos,0.2700
2,regresor + umbrales por cuantiles,0.3918


## Submission: los mismos umbrales, calibrados en out-of-fold, aplicados a la predicción de test

In [ ]:
# promedio de las 5 predicciones continuas de test, cortado con los MISMOS umbrales calibrados en out-of-fold
submission = pd.Series(cortar(test_preds.mean(axis=1), umbrales), index=X_test.index, name=label)
submission.to_csv("submission.csv")
print(submission.value_counts(normalize=True).sort_index().round(3))
submission.head()

## Tabla de resultados del laboratorio

Completar a mano después de correr y enviar cada versión.

| Versión | Qué agrega | Kappa OOF | LB público | Notas |
|---|---|---|---|---|
| v0 | Baseline de la clase 1: LightGBM sobre numéricas, KFold | | | |
| v1 | Validación por grupo + 3 features de la consigna | | | |
| v2 | Grid search de LightGBM elegido por kappa out-of-fold | | | |
| v3 | Regresión + umbrales por cuantiles | | | ← esta notebook |
| v4 | Texto: TF-IDF + SVD y estadísticas de la descripción | | | |
| v5 | Embeddings multilingües de la descripción | | | |

## Opcional, para investigar: optimizar los umbrales directamente sobre kappa

`OptimizedRounder` apareció en los notebooks públicos de esta competencia. Minimiza `-kappa` moviendo los cuatro umbrales con Nelder-Mead, arrancando desde los ingenuos. Suele terminar cerca de los cuantiles y puede sobreajustar un poco a las predicciones out-of-fold. Comparar con la fila anterior.

In [12]:
from scipy.optimize import minimize


def _perdida(u, pred, y_true):
    return -kappa(y_true, cortar(pred, u))


res = minimize(_perdida, x0=ingenuos, args=(valid_preds.values, y_train.values), method="Nelder-Mead", options={"maxiter": 500})
optimizados = np.sort(res.x)
print("umbrales optimizados:", optimizados.round(3), f"-> kappa OOF {kappa(y_train, cortar(valid_preds, optimizados)):.4f}")
print("umbrales cuantiles  :", umbrales.round(3), f"-> kappa OOF {kappa_cuantiles:.4f}")

umbrales optimizados: [0.392 2.15  2.497 2.841] -> kappa OOF 0.3954
umbrales cuantiles  : [1.751 2.135 2.466 2.8  ] -> kappa OOF 0.3920
